Basicamente iremos pegar o nosso dado que está na landing e passar para a camada bronze no formato delta e salvar como uma table, e adiconar o registro de hora de ingestão e criar um tabela chamada  bronze.dm_cotacao_dolar
que vai ser consumida de um API


In [0]:
from pyspark.sql import functions as F

In [0]:
catalogo = "ecommerce"

In [0]:
def transform_data(table_name: str,file_name: str):
  try:
    file_path = f"/Volumes/ecommerce/landing/landing/{file_name}"
    df = spark.read.csv(file_path, header=True, inferSchema=True)
    df_metadata = df.withColumn("ingestion_timestamp", F.current_timestamp())
    df_metadata.write.format("delta").saveAsTable(f"{catalogo}.{table_name}")
  except Exception as e:
    print(e)
    


Agora irei fazer a outra tabela que irá consumir do endpoint, primeiramente usando request e o pandas

In [0]:
import requests
import pandas as pd

In [0]:
## DATA INICIO FORMATA E DATA_FIM_FORMATADA COMO VÁRIAVEIS DO NOTEBOOK
data_inicio_formatada = "01-01-2016"
data_fim_formatada = "07-11-2018"

In [0]:
url = (
    f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    f"CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)"
    f"?@dataInicial='{data_inicio_formatada}'"
    f"&@dataFinalCotacao='{data_fim_formatada}'"
    f"&$select=dataHoraCotacao,cotacaoCompra&$format=json"
)


In [0]:
response = requests.get(url)

data = response.json()

In [0]:
#entendendo melhor os itens presentes no meu json

for key, value in data.items():
    if isinstance(value, list):
        for item in value:
            print(key, item)



In [0]:
pandas_df = pd.DataFrame(data)

pandas_df.head()

In [0]:
#AQUI EU PERCEBI QUE AS INFORMAÇÕES DE COLUNA ESTÃO REPETIDAS E EM UM FORMATO JSON, MAS UMA DÚVIDA EU MUDARIA ESSES DADOS NA CAMADA BRONZE?

#Irei fazer isso na bronze mesmo, porque tem um print de como as colunas deveria ficar na bronze e na silver

df = spark.createDataFrame(pandas_df)

display(df)

In [0]:
df= df.drop("@odata.context")

df.schema

In [0]:
#AGORA IREI FAZER O TRATAMENTO DO DF PARA FICAR NO FORMATO QUE EU DESEJO

df_flat = df.select(
    F.col("value.cotacaoCompra").alias("cotacaoCompra"),
    F.col("value.dataHoraCotacao").alias("dataHoraCotacao")
)
df_flat.display()


In [0]:
display(df_flat)

In [0]:
df_cotacao = (
    df_flat.withColumn("data_ingestao", F.current_timestamp())
)

In [0]:
df_cotacao.display()

In [0]:
%sql
DROP TABLE IF EXISTS ecommerce.bronze.dm_categoria_produtos_traducao;
DROP TABLE IF EXISTS ecommerce.bronze.ft_avaliacoes_pedidos;
DROP TABLE IF EXISTS ecommerce.bronze.ft_consumidores;
DROP TABLE IF EXISTS ecommerce.bronze.ft_geolocalizacao;
DROP TABLE IF EXISTS ecommerce.bronze.ft_itens_pedidos;
DROP TABLE IF EXISTS ecommerce.bronze.ft_pagamentos_pedidos;
DROP TABLE IF EXISTS ecommerce.bronze.ft_pedidos;
DROP TABLE IF EXISTS ecommerce.bronze.ft_produtos;
DROP TABLE IF EXISTS ecommerce.bronze.ft_vendedores;
DROP TABLE IF EXISTS ecommerce.bronze.dm_cotacao_dolar


In [0]:
transform_data("bronze.ft_consumidores","olist_customers_dataset.csv")
transform_data("bronze.ft_geolocalizacao","olist_geolocation_dataset.csv")
transform_data("bronze.ft_itens_pedidos","olist_order_items_dataset.csv")
transform_data("bronze.ft_pagamentos_pedidos","olist_order_payments_dataset.csv")
transform_data("bronze.ft_avaliacoes_pedidos","olist_order_reviews_dataset.csv")
transform_data("bronze.ft_pedidos","olist_orders_dataset.csv")
transform_data("bronze.ft_produtos","olist_products_dataset.csv")
transform_data("bronze.ft_vendedores","olist_sellers_dataset.csv")
transform_data("bronze.dm_categoria_produtos_traducao","product_category_name_translation.csv")
df_cotacao.write.format("delta").saveAsTable(f"{catalogo}.bronze.dm_cotacao_dolar")


TIVE QUE MUDAR AS DATAS DAS COTAÇÕES POIS ELAS ERAM MUITO ATUAIS E NÃO ESTAVA FAZENDO UM MATCH EM NENHUMA NO MEU OUTRO EXERCICIO